In [1]:
import pandas as pd 
import numpy as np
import joblib
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import(
     RandomForestClassifier,
     ExtraTreesClassifier,
     AdaBoostClassifier,
     GradientBoostingClassifier,
     StackingClassifier
)
from sklearn.metrics import (
     accuracy_score, 
     precision_score, 
     recall_score, 
     f1_score, 
     roc_auc_score, 
     confusion_matrix, 
     ConfusionMatrixDisplay)

In [2]:
df=pd.read_csv("data\heart_disease_uci.csv")
df.columns=df.columns.str.strip()

<>:1: SyntaxWarning: invalid escape sequence '\h'
<>:1: SyntaxWarning: invalid escape sequence '\h'
C:\Users\USER\AppData\Local\Temp\ipykernel_10884\1459453880.py:1: SyntaxWarning: invalid escape sequence '\h'
  df=pd.read_csv("data\heart_disease_uci.csv")


In [3]:
print(df.head())
print(df.info())

   id  age     sex    dataset               cp  trestbps   chol    fbs  \
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True   
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False   
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False   
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False   
4   5   41  Female  Cleveland  atypical angina     130.0  204.0  False   

          restecg  thalch  exang  oldpeak        slope   ca  \
0  lv hypertrophy   150.0  False      2.3  downsloping  0.0   
1  lv hypertrophy   108.0   True      1.5         flat  3.0   
2  lv hypertrophy   129.0   True      2.6         flat  2.0   
3          normal   187.0  False      3.5  downsloping  0.0   
4  lv hypertrophy   172.0  False      1.4    upsloping  0.0   

                thal  num  
0       fixed defect    0  
1             normal    2  
2  reversable defect    1  
3             normal    0  
4             normal    0  
<class '

In [4]:
print(df.isnull().sum())
print("Duplicate Rows :", df.duplicated().sum())
df.drop(columns="id", inplace=True)
df["num"] = df["num"].apply(lambda x: 0 if x == 0 else 1)
numerical_columns = df.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()
print(numerical_columns)
print(categorical_columns)
for col in numerical_columns:
    df[col]=df[col].fillna(df[col].median())
for col in categorical_columns:
    df[col]=df[col].fillna(df[col].mode()[0])
print("Missing values after filling")
print(df.isnull().sum())

id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64
Duplicate Rows : 0
['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'num']
['sex', 'dataset', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
Missing values after filling
age         0
sex         0
dataset     0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
num         0
dtype: int64


C:\Users\USER\AppData\Local\Temp\ipykernel_10884\4187393789.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()


In [5]:
encoders={}
categorical_cols=[
    "sex",
    "dataset",                
    "cp",         
    "fbs",      
    "restecg",                 
    "exang", 
    "slope", 
    "thal"  
]
for col in categorical_cols:
    encoder=LabelEncoder()
    df[col]=encoder.fit_transform(df[col])
    encoders[col]=encoder
joblib.dump(encoders,"encoder.pkl")
print(df.head())

   age  sex  dataset  cp  trestbps   chol  fbs  restecg  thalch  exang  \
0   63    1        0   3     145.0  233.0    1        0   150.0      0   
1   67    1        0   0     160.0  286.0    0        0   108.0      1   
2   67    1        0   0     120.0  229.0    0        0   129.0      1   
3   37    1        0   2     130.0  250.0    0        1   187.0      0   
4   41    0        0   1     130.0  204.0    0        0   172.0      0   

   oldpeak  slope   ca  thal  num  
0      2.3      0  0.0     0    0  
1      1.5      1  3.0     1    1  
2      2.6      1  2.0     2    1  
3      3.5      0  0.0     1    0  
4      1.4      2  0.0     1    0  


In [6]:
x=df.drop("num",axis=1)
y=df["num"]

In [7]:
scaler=StandardScaler()
x=scaler.fit_transform(x)
joblib.dump(scaler,"scaler.pkl")

['scaler.pkl']

In [8]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [9]:
models={
    "Logistic Regression":LogisticRegression(max_iter=1000,random_state=42),
    "KNN":KNeighborsClassifier(),
    "SVM":SVC(
    C=1,
    kernel="rbf",
    gamma="scale",
    probability=True,
    random_state=42
),
    "Decision Tree":DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
),
    "Random Forest":RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    min_samples_split=10,
    max_features="sqrt",
    random_state=42
),
    "Extra Trees":ExtraTreesClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    random_state=42
),
    "AdaBoost":AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42
),
    "Gradient Boosting":GradientBoostingClassifier(
    learning_rate=0.05,
    n_estimators=100,
    max_depth=3,
    random_state=42
)
}

In [10]:
results = []
trained_models = {}

for name, model in models.items():

    model.fit(x_train, y_train)

    trained_models[name] = model

    train_pred = model.predict(x_train)
    test_pred = model.predict(x_test)

    results.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Test Accuracy": accuracy_score(y_test, test_pred)
    })

results_df = pd.DataFrame(results)
display(results_df)

C:\Users\USER\Desktop\ML\all projects\intermidiate\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Train Accuracy,Test Accuracy
0,Logistic Regression,0.815217,0.820652
1,KNN,0.839674,0.842391
2,SVM,0.875000,0.842391
3,Decision Tree,0.842391,0.809783
4,Random Forest,0.876359,0.842391
5,Extra Trees,0.932065,0.826087
6,AdaBoost,0.838315,0.842391
7,Gradient Boosting,0.880435,0.836957


In [11]:
estimators = [
    ("rf", trained_models["Random Forest"]),
    ("et", trained_models["Extra Trees"]),
    ("gb", trained_models["Gradient Boosting"]),
    ("svm", trained_models["SVM"])
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

stack_model.fit(x_train, y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.The type of estimator is generally expected to be a classifier.However, one can pass a regressor for some use case (e.g. ordinalregression).","[('rf', ...), ('et', ...), ...]"
,"final_estimator final_estimator: estimator, default=NoneA classifier which will be used to combine the base estimators.The default classifier is a:class:`~sklearn.linear_model.LogisticRegression`.",LogisticRegre...max_iter=1000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"stack_method stack_method: {'auto', 'predict_proba', 'decision_function', 'predict'}, default='auto'Methods called for each base estimator. It can be:* if 'auto', it will try to invoke, for each estimator, `'predict_proba'`, `'decision_function'` or `'predict'` in that order.* otherwise, one of `'predict_proba'`, `'decision_function'` or `'predict'`. If the method is not implemented by the estimator, it will raise an error.",'auto'
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,) or list of ndarray if `y` is of type `""multilabel-indicator""`.Class labels.","ndarray[int64](2,)","[0,1]"
"estimators_ estimators_: list of estimatorsThe elements of the `estimators` parameter, having been fitted on thetraining data. If an estimator has been set to `'drop'`, itwill not appear in `estimators_`. When `cv=""prefit""`, `estimators_`is set to `estimators` and is not fitted again.",list,"[RandomForestC...ndom_state=42), ExtraTreesCla...ndom_state=42), GradientBoost...ndom_state=42), SVC(C=1, prob...ndom_state=42)]"
final_estimator_ final_estimator_: estimatorThe classifier fit on the output of `estimators_` and responsible forfinal predictions.,LogisticRegression,LogisticRegre...max_iter=1

In [12]:
train_pred = stack_model.predict(x_train)
test_pred = stack_model.predict(x_test)

print("Train Accuracy :", accuracy_score(y_train, train_pred))
print("Test Accuracy  :", accuracy_score(y_test, test_pred))
print("Precision      :", precision_score(y_test, test_pred))
print("Recall         :", recall_score(y_test, test_pred))
print("F1 Score       :", f1_score(y_test, test_pred))

if hasattr(stack_model, "predict_proba"):
    prob = stack_model.predict_proba(x_test)[:, 1]
    print("ROC AUC        :", roc_auc_score(y_test, prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,test_pred))

Train Accuracy : 0.8953804347826086
Test Accuracy  : 0.842391304347826
Precision      : 0.8411214953271028
Recall         : 0.8823529411764706
F1 Score       : 0.861244019138756
ROC AUC        : 0.9289813486370158

Confusion Matrix
[[65 17]
 [12 90]]


In [13]:
joblib.dump(stack_model, "heart_disease_stacking_model.pkl")
print("Model saved successfully!")

Model saved successfully!
